# 03 — Evaluation: FID, IS, Runtime

Run this after training to compute all metrics required by `ARCHITECTURE.md`.

**What this notebook does:**
1. Loads a trained checkpoint from Google Drive
2. Generates images at NFE = 10, 20, 50, 100 (DDIM) and 1000 (DDPM)
3. Computes FID and IS for each
4. Measures runtime (sec/image)
5. Saves all results to Drive in the shared schema from `ARCHITECTURE.md`

**Run on Colab with GPU.**

In [ ]:
import math
import time
import json
import os
import subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from dataclasses import dataclass
from typing import Tuple

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Auto-install torchmetrics if missing
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.inception import InceptionScore
except ImportError:
    print('Installing torchmetrics[image]...')
    subprocess.check_call(['pip', 'install', '-q', 'torchmetrics[image]'])
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.inception import InceptionScore

print('torchmetrics ready — FID and IS will be computed.')

In [ ]:
# --- Mount Google Drive ---
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    print('Google Drive mounted.')
except ModuleNotFoundError:
    print('Not on Colab — skipping Drive mount.')

# --- Paths (must match colab_setup.ipynb and 02_ddpm.ipynb) ---
DRIVE_DIR       = '/content/drive/MyDrive/flow-matching'
DDPM_RUN_DIR    = f'{DRIVE_DIR}/runs/ddpm'
METRICS_DIR     = f'{DDPM_RUN_DIR}/metrics'
EVAL_IMAGES_DIR = f'{DDPM_RUN_DIR}/samples/eval'

os.makedirs(METRICS_DIR,     exist_ok=True)
os.makedirs(EVAL_IMAGES_DIR, exist_ok=True)

# NFE points to evaluate — matches ARCHITECTURE.md requirement
NFE_POINTS = [10, 20, 50, 100]

# Number of images to generate for FID (10k is fast, 50k is more accurate)
# Use 10000 for development, 50000 for final results
NUM_EVAL_IMAGES = 10000
EVAL_BATCH_SIZE = 256
EVAL_SEED       = 42

print(f'DDPM run dir : {DDPM_RUN_DIR}')
print(f'Metrics dir  : {METRICS_DIR}')
print(f'NFE points   : {NFE_POINTS}')
print(f'Num images   : {NUM_EVAL_IMAGES}')

---
## Step 1: Copy model definitions from 02_ddpm.ipynb

We need all the model classes here too. In the future these will live in `src/` as proper modules — for now we re-define them.

In [ ]:
# ============================================================
# CONFIG
# ============================================================
@dataclass
class DDPMConfig:
    image_size: int = 32
    channels: int = 3
    T: int = 1000
    schedule: str = 'cosine'
    beta_start: float = 1e-4
    beta_end: float = 0.02
    model_channels: int = 128
    num_heads: int = 4
    dropout: float = 0.1
    num_groups: int = 32
    batch_size: int = 128
    lr: float = 2e-4
    num_epochs: int = 500
    grad_clip: float = 1.0
    ema_decay: float = 0.9999
    clip_denoised: bool = True

cfg = DDPMConfig()

# ============================================================
# NOISE SCHEDULE
# ============================================================
def cosine_schedule(T, s=0.008):
    steps = T + 1
    t = torch.linspace(0, T, steps) / T
    f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    alphas_cumprod = f / f[0]
    betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1]
    return betas.clamp(0, 0.999)

def linear_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

class NoiseScheduler:
    def __init__(self, cfg):
        self.T = cfg.T
        betas = cosine_schedule(cfg.T) if cfg.schedule == 'cosine' else linear_schedule(cfg.T, cfg.beta_start, cfg.beta_end)
        alphas = 1.0 - betas
        alphas_cumprod      = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
        self.betas                = betas
        self.alphas_cumprod       = alphas_cumprod
        self.alphas_cumprod_prev  = alphas_cumprod_prev
        self.sqrt_alphas_cumprod  = alphas_cumprod.sqrt()
        self.sqrt_one_minus_ac    = (1 - alphas_cumprod).sqrt()
        self.sqrt_recip_alphas    = (1.0 / alphas).sqrt()
        self.posterior_variance   = betas * (1 - alphas_cumprod_prev) / (1 - alphas_cumprod)
        # Posterior mean coefficients: \u03bc\u0303\u209c = coeff1 \u00b7 x\u0302\u2080 + coeff2 \u00b7 x\u209c
        self.posterior_mean_coeff1 = (alphas_cumprod_prev.sqrt() * betas) / (1 - alphas_cumprod)
        self.posterior_mean_coeff2 = (alphas.sqrt() * (1 - alphas_cumprod_prev)) / (1 - alphas_cumprod)

# ============================================================
# MODEL
# ============================================================
class TimeEmbedding(nn.Module):
    def __init__(self, base_dim):
        super().__init__()
        self.base_dim = base_dim
        self.mlp = nn.Sequential(nn.Linear(base_dim, base_dim*4), nn.SiLU(), nn.Linear(base_dim*4, base_dim*4))
    def forward(self, t):
        half  = self.base_dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        emb   = t[:, None].float() * freqs[None]
        emb   = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return self.mlp(emb)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_groups=32, dropout=0.1):
        super().__init__()
        self.norm1    = nn.GroupNorm(num_groups, in_ch)
        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch * 2))
        self.norm2    = nn.GroupNorm(num_groups, out_ch)
        self.dropout  = nn.Dropout(dropout)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act      = nn.SiLU()
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        scale, shift = self.time_mlp(t_emb).chunk(2, dim=-1)
        h = h * (scale[:, :, None, None] + 1) + shift[:, :, None, None]
        h = self.dropout(self.conv2(self.act(self.norm2(h))))
        return h + self.shortcut(x)

class SelfAttention(nn.Module):
    def __init__(self, channels, num_heads=4, num_groups=32):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = channels // num_heads
        self.norm      = nn.GroupNorm(num_groups, channels)
        self.to_qkv    = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.to_out    = nn.Conv2d(channels, channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        h   = self.norm(x)
        qkv = self.to_qkv(h).reshape(B, 3, self.num_heads, self.head_dim, H * W).permute(1, 0, 2, 4, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = F.scaled_dot_product_attention(q, k, v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        return x + self.to_out(attn)

class Downsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)
    def forward(self, x): return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)
    def forward(self, x): return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))

class UNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        ch, t_dim, ng, nh, do = cfg.model_channels, cfg.model_channels*4, cfg.num_groups, cfg.num_heads, cfg.dropout
        self.time_emb = TimeEmbedding(ch)
        self.init_conv = nn.Conv2d(3, ch, 3, padding=1)
        self.enc1_a = ResBlock(ch,   ch,   t_dim, ng, do); self.enc1_b = ResBlock(ch,   ch,   t_dim, ng, do); self.down1 = Downsample(ch)
        self.enc2_a = ResBlock(ch,   ch*2, t_dim, ng, do); self.enc2_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.attn2 = SelfAttention(ch*2, nh, ng); self.down2 = Downsample(ch*2)
        self.enc3_a = ResBlock(ch*2, ch*2, t_dim, ng, do); self.enc3_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.attn3 = SelfAttention(ch*2, nh, ng); self.down3 = Downsample(ch*2)
        self.mid1 = ResBlock(ch*2, ch*2, t_dim, ng, do); self.mid_attn = SelfAttention(ch*2, nh, ng); self.mid2 = ResBlock(ch*2, ch*2, t_dim, ng, do)
        self.up3 = Upsample(ch*2); self.dec3_a = ResBlock(ch*4, ch*2, t_dim, ng, do); self.dec3_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.dattn3 = SelfAttention(ch*2, nh, ng)
        self.up2 = Upsample(ch*2); self.dec2_a = ResBlock(ch*4, ch*2, t_dim, ng, do); self.dec2_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.dattn2 = SelfAttention(ch*2, nh, ng)
        self.up1 = Upsample(ch*2); self.dec1_a = ResBlock(ch*3, ch,   t_dim, ng, do); self.dec1_b = ResBlock(ch,   ch,   t_dim, ng, do)
        self.out_norm = nn.GroupNorm(ng, ch); self.out_conv = nn.Conv2d(ch, 3, 1)
    def forward(self, x, t):
        t_emb = self.time_emb(t)
        h = self.init_conv(x)
        h = self.enc1_b(self.enc1_a(h, t_emb), t_emb);  s1 = h
        h = self.attn2(self.enc2_b(self.enc2_a(self.down1(h), t_emb), t_emb)); s2 = h
        h = self.attn3(self.enc3_b(self.enc3_a(self.down2(h), t_emb), t_emb)); s3 = h
        h = self.down3(h); h = self.mid1(h, t_emb); h = self.mid_attn(h); h = self.mid2(h, t_emb)
        h = self.dattn3(self.dec3_b(self.dec3_a(torch.cat([self.up3(h), s3], dim=1), t_emb), t_emb))
        h = self.dattn2(self.dec2_b(self.dec2_a(torch.cat([self.up2(h), s2], dim=1), t_emb), t_emb))
        h = self.dec1_b(self.dec1_a(torch.cat([self.up1(h), s1], dim=1), t_emb), t_emb)
        return self.out_conv(F.silu(self.out_norm(h)))

print('Model classes loaded.')

---
## Step 2: Load checkpoint from Drive

In [ ]:
def load_model_from_drive(checkpoint_name: str = 'last.pt', use_ema: bool = True):
    """Load trained model from Drive checkpoint."""
    path = f'{DDPM_RUN_DIR}/checkpoints/{checkpoint_name}'
    assert os.path.exists(path), f'Checkpoint not found: {path}'

    ckpt  = torch.load(path, map_location=device)
    model = UNet(cfg).to(device)

    if use_ema and 'ema_shadow' in ckpt:
        for name, param in model.named_parameters():
            param.data.copy_(ckpt['ema_shadow'][name])
        print(f'Loaded EMA weights from {checkpoint_name} (epoch {ckpt["epoch"]}, loss={ckpt["loss"]:.4f})')
    else:
        model.load_state_dict(ckpt['model'])
        print(f'Loaded model weights from {checkpoint_name} (epoch {ckpt["epoch"]})')

    model.eval()
    return model, ckpt


model, checkpoint = load_model_from_drive('last.pt', use_ema=True)
scheduler = NoiseScheduler(cfg)
print(f'Ready to evaluate.')

---
## Step 3: Samplers

In [ ]:
@torch.no_grad()
def ddpm_sample(model, scheduler, shape, device, clip_denoised=True):
    """Full DDPM reverse process — predict x\u0302\u2080, clip that, then compute posterior mean."""
    model.eval()
    x = torch.randn(shape, device=device)
    for t in tqdm(reversed(range(scheduler.T)), desc='DDPM', total=scheduler.T, leave=False):
        t_batch = torch.full((shape[0],), t, device=device, dtype=torch.long)
        eps_hat = model(x, t_batch)
        # Predict x\u0302\u2080 and clip the clean image prediction (not the noisy mean!)
        x0_pred = (x - scheduler.sqrt_one_minus_ac[t] * eps_hat) / scheduler.sqrt_alphas_cumprod[t]
        if clip_denoised:
            x0_pred = x0_pred.clamp(-1, 1)
        # Posterior mean from x\u0302\u2080
        mean = scheduler.posterior_mean_coeff1[t] * x0_pred + scheduler.posterior_mean_coeff2[t] * x
        if t > 0:
            x = mean + scheduler.posterior_variance[t].sqrt() * torch.randn_like(x)
        else:
            x = mean
    return x

@torch.no_grad()
def ddim_sample(model, scheduler, shape, device, num_steps=50, clip_denoised=True):
    model.eval()
    step  = scheduler.T // num_steps
    steps = list(reversed(range(0, scheduler.T, step)))
    x     = torch.randn(shape, device=device)
    for i, t in enumerate(tqdm(steps, desc=f'DDIM-{num_steps}', leave=False)):
        t_batch = torch.full((shape[0],), t, device=device, dtype=torch.long)
        t_prev  = steps[i + 1] if i + 1 < len(steps) else 0
        eps_hat = model(x, t_batch)
        ac      = scheduler.alphas_cumprod[t].to(device)
        ac_prev = scheduler.alphas_cumprod[t_prev].to(device) if t_prev > 0 else torch.tensor(1.0, device=device)
        x0_pred = ((x - (1 - ac).sqrt() * eps_hat) / ac.sqrt()).clamp(-1, 1) if clip_denoised else (x - (1 - ac).sqrt() * eps_hat) / ac.sqrt()
        x       = ac_prev.sqrt() * x0_pred + (1 - ac_prev).sqrt() * eps_hat
    return x

print('Samplers ready.')

---
## Step 4: Generate images + compute metrics

For each NFE point we:
1. Generate `NUM_EVAL_IMAGES` images in batches (timed)
2. Compute FID against real CIFAR-10
3. Compute IS
4. Save results to Drive in the ARCHITECTURE.md schema

In [ ]:
import gc

def get_real_images(n: int, batch_size: int = 256):
    """Load n real CIFAR-10 images as uint8 tensors [N, 3, 32, 32] in [0, 255]."""
    dataset = datasets.CIFAR10(
        root='./data', train=True, download=True,
        transform=transforms.ToTensor()
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    imgs = []
    for x, _ in tqdm(loader, desc='Loading real images'):
        imgs.append((x * 255).byte())
        if sum(b.shape[0] for b in imgs) >= n:
            break
    return torch.cat(imgs)[:n]


def generate_images(model, scheduler, n: int, batch_size: int,
                    sampler: str = 'ddim', num_steps: int = 50,
                    seed: int = 42):
    """Generate n images in batches. Returns uint8 [N, 3, 32, 32] on CPU and total time."""
    torch.manual_seed(seed)
    all_images = []
    total_time = 0.0
    generated  = 0

    while generated < n:
        bs = min(batch_size, n - generated)

        t_start = time.time()
        if sampler == 'ddpm':
            imgs = ddpm_sample(model, scheduler, (bs, 3, 32, 32), device)
        else:
            imgs = ddim_sample(model, scheduler, (bs, 3, 32, 32), device, num_steps=num_steps)
        total_time += time.time() - t_start

        # Move to CPU as uint8 immediately to free GPU memory
        imgs = ((imgs.cpu() + 1) / 2).clamp(0, 1)
        imgs = (imgs * 255).byte()
        all_images.append(imgs)
        generated += bs

        # Free GPU memory between batches
        torch.cuda.empty_cache()

    return torch.cat(all_images)[:n], total_time


print('Image generation helpers ready.')
print(f'Will generate {NUM_EVAL_IMAGES} images per DDIM NFE point.')

In [ ]:
def compute_fid_is(real_imgs: torch.Tensor, fake_imgs: torch.Tensor):
    """Compute FID and IS. Both inputs are uint8 [N, 3, 32, 32].
    Deletes metric objects after use to free GPU memory.
    """
    bs = 256

    # FID
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)
    for i in range(0, len(real_imgs), bs):
        fid_metric.update(real_imgs[i:i+bs].to(device), real=True)
    for i in range(0, len(fake_imgs), bs):
        fid_metric.update(fake_imgs[i:i+bs].to(device), real=False)
    fid_score = fid_metric.compute().item()
    del fid_metric

    # IS
    is_metric = InceptionScore().to(device)
    for i in range(0, len(fake_imgs), bs):
        is_metric.update(fake_imgs[i:i+bs].to(device))
    is_mean, is_std = is_metric.compute()
    is_mean, is_std = is_mean.item(), is_std.item()
    del is_metric

    torch.cuda.empty_cache()
    return fid_score, is_mean, is_std


print('Metric computation ready.')

In [ ]:
# Load real images once (kept on CPU — small: 10k * 3 * 32 * 32 = ~30MB)
print('Loading real CIFAR-10 images...')
real_imgs = get_real_images(NUM_EVAL_IMAGES)
print(f'Real images: {real_imgs.shape}, dtype={real_imgs.dtype}')

all_results = []
checkpoint_name = os.path.basename(f'{DDPM_RUN_DIR}/checkpoints/last.pt')

# ---------------------------------------------------------------
# Evaluate DDIM at each NFE point (one at a time, free memory between)
# ---------------------------------------------------------------
for nfe in NFE_POINTS:
    print(f'\n{"="*50}')
    print(f'Evaluating DDIM at NFE={nfe}')
    print(f'{"="*50}')

    fake_imgs, total_sec = generate_images(
        model, scheduler, n=NUM_EVAL_IMAGES,
        batch_size=EVAL_BATCH_SIZE, sampler='ddim',
        num_steps=nfe, seed=EVAL_SEED
    )

    # Save sample grid BEFORE computing metrics (in case it crashes)
    nfe_dir = f'{EVAL_IMAGES_DIR}/nfe_{nfe}'
    os.makedirs(nfe_dir, exist_ok=True)
    grid = torchvision.utils.make_grid(fake_imgs[:64].float() / 255, nrow=8, padding=2)
    torchvision.utils.save_image(grid, f'{nfe_dir}/sample_grid.png')

    # Compute metrics
    fid, is_mean, is_std = compute_fid_is(real_imgs, fake_imgs)
    sec_per_image = total_sec / NUM_EVAL_IMAGES

    result = {
        'model': 'ddpm', 'checkpoint_name': checkpoint_name,
        'nfe': nfe, 'solver_sampler': 'ddim',
        'num_generated': NUM_EVAL_IMAGES,
        'fid': round(fid, 4), 'is_mean': round(is_mean, 4), 'is_std': round(is_std, 4),
        'sec_per_image': round(sec_per_image, 6),
        'total_sampling_sec': round(total_sec, 2),
        'seed': EVAL_SEED, 'notes': f'epoch {checkpoint["epoch"]}',
    }
    all_results.append(result)
    print(f'  FID={fid:.2f}  IS={is_mean:.2f}\u00b1{is_std:.2f}  {sec_per_image*1000:.1f}ms/img')

    # FREE memory before next NFE
    del fake_imgs
    torch.cuda.empty_cache()
    gc.collect()

# ---------------------------------------------------------------
# DDPM 1000 steps — same quality settings, memory cleaned between batches
# ---------------------------------------------------------------
print(f'\n{"="*50}')
print(f'Evaluating DDPM at NFE=1000 ({NUM_EVAL_IMAGES} images)')
print(f'{"="*50}')

fake_imgs_ddpm, total_sec_ddpm = generate_images(
    model, scheduler, n=NUM_EVAL_IMAGES,
    batch_size=EVAL_BATCH_SIZE, sampler='ddpm',
    num_steps=1000, seed=EVAL_SEED
)

# Save sample grid
nfe_dir = f'{EVAL_IMAGES_DIR}/nfe_1000'
os.makedirs(nfe_dir, exist_ok=True)
grid = torchvision.utils.make_grid(fake_imgs_ddpm[:64].float() / 255, nrow=8, padding=2)
torchvision.utils.save_image(grid, f'{nfe_dir}/sample_grid.png')

fid, is_mean, is_std = compute_fid_is(real_imgs, fake_imgs_ddpm)
sec_per_image = total_sec_ddpm / NUM_EVAL_IMAGES

result_ddpm = {
    'model': 'ddpm', 'checkpoint_name': checkpoint_name,
    'nfe': 1000, 'solver_sampler': 'ddpm',
    'num_generated': NUM_EVAL_IMAGES,
    'fid': round(fid, 4), 'is_mean': round(is_mean, 4), 'is_std': round(is_std, 4),
    'sec_per_image': round(sec_per_image, 6),
    'total_sampling_sec': round(total_sec_ddpm, 2),
    'seed': EVAL_SEED, 'notes': f'epoch {checkpoint["epoch"]}',
}
all_results.append(result_ddpm)
print(f'  FID={fid:.2f}  IS={is_mean:.2f}\u00b1{is_std:.2f}  {sec_per_image*1000:.1f}ms/img')

del fake_imgs_ddpm
torch.cuda.empty_cache()
gc.collect()

print('\nAll evaluations done!')

---
## Step 5: Save metrics to Drive

In [ ]:
import datetime

# Save full results list as JSONL (one entry per line — easy to append later)
jsonl_path = f'{METRICS_DIR}/metrics_summary.jsonl'
with open(jsonl_path, 'a') as f:   # 'a' = append, so multiple eval runs accumulate
    for r in all_results:
        r['evaluated_at'] = datetime.datetime.now().isoformat()
        f.write(json.dumps(r) + '\n')

# Save latest run as clean JSON summary
json_path = f'{METRICS_DIR}/metrics_summary.json'
with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)

# Save FID-specific results
fid_results = [{'nfe': r['nfe'], 'solver_sampler': r['solver_sampler'],
                'fid': r['fid'], 'is_mean': r['is_mean'], 'is_std': r['is_std']} for r in all_results]
with open(f'{METRICS_DIR}/fid_results.json', 'w') as f:
    json.dump(fid_results, f, indent=2)

# Save runtime results
runtime_results = [{'nfe': r['nfe'], 'solver_sampler': r['solver_sampler'],
                    'sec_per_image': r['sec_per_image'],
                    'total_sampling_sec': r['total_sampling_sec']} for r in all_results]
with open(f'{METRICS_DIR}/runtime_results.json', 'w') as f:
    json.dump(runtime_results, f, indent=2)

print(f'Metrics saved to {METRICS_DIR}')
print(f'  metrics_summary.json   — full results')
print(f'  metrics_summary.jsonl  — appendable log')
print(f'  fid_results.json       — FID + IS per NFE')
print(f'  runtime_results.json   — timing per NFE')

---
## Step 6: Visualize results

In [ ]:
# FID vs NFE curve
ddim_results = [r for r in all_results if r['solver_sampler'] == 'ddim']
ddpm_result  = [r for r in all_results if r['solver_sampler'] == 'ddpm'][0]

nfes = [r['nfe'] for r in ddim_results]
fids = [r['fid'] for r in ddim_results]
spis = [r['sec_per_image'] * 1000 for r in ddim_results]  # ms/img

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('DDPM/DDIM Evaluation Results', fontsize=13)

# FID vs NFE
axes[0].plot(nfes, fids, 'o-', color='steelblue', linewidth=2, markersize=8, label='DDIM')
axes[0].axhline(ddpm_result['fid'], color='tomato', linestyle='--', linewidth=2, label=f'DDPM (1000 steps) FID={ddpm_result["fid"]:.1f}')
axes[0].set_xlabel('NFE (number of function evaluations)')
axes[0].set_ylabel('FID \u2193 (lower is better)')
axes[0].set_title('FID vs NFE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Runtime vs NFE
axes[1].plot(nfes, spis, 's-', color='seagreen', linewidth=2, markersize=8, label='DDIM')
axes[1].axhline(ddpm_result['sec_per_image'] * 1000, color='tomato', linestyle='--',
                linewidth=2, label=f'DDPM (1000 steps) {ddpm_result["sec_per_image"]*1000:.1f}ms')
axes[1].set_xlabel('NFE')
axes[1].set_ylabel('ms / image \u2193')
axes[1].set_title('Runtime vs NFE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{METRICS_DIR}/fid_vs_nfe.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to Drive.')

In [ ]:
# Show sample grids side by side for each NFE
print('Sample grids at each NFE:')
for nfe in NFE_POINTS:
    grid_path = f'{EVAL_IMAGES_DIR}/nfe_{nfe}/sample_grid.png'
    if os.path.exists(grid_path):
        img = plt.imread(grid_path)
        plt.figure(figsize=(10, 3))
        plt.imshow(img)
        plt.title(f'DDIM \u2014 NFE={nfe}')
        plt.axis('off')
        plt.show()

In [ ]:
# Print final results table — matches ARCHITECTURE.md schema
print(f'\n{"Model":<8} {"Sampler":<8} {"NFE":<6} {"FID":<8} {"IS":<12} {"ms/img":<10}')
print('-' * 56)
for r in sorted(all_results, key=lambda x: x['nfe']):
    fid_str = f'{r["fid"]:.2f}' if r['fid'] else 'N/A'
    is_str  = f'{r["is_mean"]:.2f}\u00b1{r["is_std"]:.2f}' if r['is_mean'] else 'N/A'
    ms_str  = f'{r["sec_per_image"]*1000:.1f}'
    print(f'{r["model"]:<8} {r["solver_sampler"]:<8} {r["nfe"]:<6} {fid_str:<8} {is_str:<12} {ms_str:<10}')